In [1]:
import pandas as pd

entities_path = "../output/entities.parquet"
entities = pd.read_parquet(entities_path)

print("Number of entities:", len(entities))
print("Columns:", entities.columns.tolist())

Number of entities: 539
Columns: ['id', 'human_readable_id', 'title', 'type', 'description', 'text_unit_ids', 'frequency', 'degree']


In [2]:
pv_candidates = entities[
    entities["title"].str.contains(
        r"PHOTOVOLTAIK|FOTOVOLTAIK|PV[- ]?ANLAGE|\bPV\b",
        case=False,
        na=False,
        regex=True
    )
].copy()

pv_candidates[
    ["human_readable_id", "title", "type", "frequency", "degree"]
].sort_values("title")

,human_readable_id,title,type,frequency,degree
216,216,ABSCHLÄGE BEI PV-FREIFLÄCHENANLAGEN,SUPPORT_SCHEME,1,3
93,93,AGRI-PV-ANLAGE,TECHNOLOGY,3,7
245,245,AGRI-PV-ANLAGEN,TECHNOLOGY,1,2
345,345,ANREIZE FÜR PV-AUSBAU,SUPPORT_SCHEME,1,1
78,78,ANTEIL PHOTOVOLTAIK AN GESAMTER ENERGIEVERSORG...,TARGET,1,2
...,...,...,...,...,...
139,139,ZEITOPTIMALE NUTZUNG VON PV-STROM,TARGET,1,1
20,20,ZIELSETZUNGEN FÜR DEN PV-AUSBAU,TARGET,1,1
303,303,ZUKÜNFTIGE ZEITWEISE RELEVANTE ÜBERKAPAZITÄTEN...,CONSTRAINT,1,2
359,359,ÖKOLOGISCHER MEHRWERT VON PV-BIODIVERSITÄTSANL...,MARKET_METRIC,1,3


In [3]:
pv_installation_candidates = entities[
    (entities["type"] == "TECHNOLOGY") &
    (
        entities["title"].str.contains(
            r"PV[- ]?ANLAG|PHOTOVOLTAIKANLAG|FOTOVOLTAIKANLAG",
            case=False,
            na=False,
            regex=True
        )
    )
].copy()

pv_installation_candidates[
    ["human_readable_id", "title", "type", "frequency", "degree"]
].sort_values("title")

,human_readable_id,title,type,frequency,degree
93,93,AGRI-PV-ANLAGE,TECHNOLOGY,3,7
245,245,AGRI-PV-ANLAGEN,TECHNOLOGY,1,2
349,349,BIODIVERSITÄTS-PV-ANLAGE,TECHNOLOGY,1,3
211,211,FOTOVOLTAIKANLAGE,TECHNOLOGY,1,16
334,334,GRÖSSERE PV-ANLAGEN,TECHNOLOGY,1,1
358,358,LANDWIRTSCHAFTLICHE DOPPELNUTZUNG (IM RAHMEN V...,TECHNOLOGY,1,1
180,180,PHOTOVOLTAIKANLAGEN,TECHNOLOGY,2,8
301,301,PHOTOVOLTAIKANLAGEN MIT OST-WEST ODER WINTEROP...,TECHNOLOGY,1,1
92,92,PV-ANLAGE,TECHNOLOGY,5,26
42,42,PV-ANLAGEN,TECHNOLOGY,3,5


In [5]:
generic_pv_ids = [211, 180, 92, 42]

generic_pv = entities[
    entities["human_readable_id"].isin(generic_pv_ids)
]

generic_pv[
    [
        "human_readable_id",
        "title",
        "type",
        "description",
        "text_unit_ids",
        "frequency",
        "degree"
    ]
]

,human_readable_id,title,type,description,text_unit_ids,frequency,degree
42,42,PV-ANLAGEN,TECHNOLOGY,PV-ANLAGEN (photovoltaic installations or PV s...,[50fb8f6506c2271985ee7ffe42523d2eb06abbd85be8e...,3,5
92,92,PV-ANLAGE,TECHNOLOGY,PV-ANLAGE (photovoltaic system) refers to a te...,[e498566a78d36b3ce13304157993b9b4269045840f1d6...,5,26
180,180,PHOTOVOLTAIKANLAGEN,TECHNOLOGY,PHOTOVOLTAIKANLAGEN refers to photovoltaic ins...,[06c273ec9919eb4693462fdacebd50536ccdcf47bacc4...,2,8
211,211,FOTOVOLTAIKANLAGE,TECHNOLOGY,A photovoltaic installation/system used for ge...,[23663988970639bea940978ccbd9150fd292856f2f31a...,1,16


In [6]:
for _, row in generic_pv.sort_values("human_readable_id").iterrows():
    print("=" * 80)
    print("ID:", row["human_readable_id"])
    print("TITLE:", row["title"])
    print("TYPE:", row["type"])
    print("DESCRIPTION:")
    print(row["description"])
    print("TEXT UNIT IDS:")
    print(row["text_unit_ids"])
    print()

ID: 42
TITLE: PV-ANLAGEN
TYPE: TECHNOLOGY
DESCRIPTION:
PV-ANLAGEN (photovoltaic installations or PV systems) are identified as the installation type whose technical, economic, and systemic development is a major focus of the Austrian PV strategy, and include small and medium-sized systems that are subject to various funding and incentive schemes; by 2040, PV systems, especially in buildings and infrastructure, are stated to have become the standard for energy generation in new and fundamentally renovated buildings.
TEXT UNIT IDS:
['50fb8f6506c2271985ee7ffe42523d2eb06abbd85be8ea5021ae2d36d260f6f63d177e59ca2da764eeaeb4a953c841282dbf214859a753f968cb6e6cb86e0c72'
 '50e06bf07ca2ce3a4e0dc0520305a36334e339dc022e32de2a78b62769102078202b01514330a1c4939e30fdb8f4d3884f790c7814b2e95e9bdb0894b3323a4e'
 'dafd7f2f39e49c87969ec1360e9fae0af9de6f290655205c8c120dd24dcc85da04f0536237f7f9529d31deed892d5abae3b8f58fb66ae9e497327f23b1e6b750']

ID: 92
TITLE: PV-ANLAGE
TYPE: TECHNOLOGY
DESCRIPTION:
PV-ANLAGE (p

In [7]:
generic_pv[
    [
        "id",
        "human_readable_id",
        "title",
        "type"
    ]
]

,id,human_readable_id,title,type
42,3b926070-fa35-4f04-ba30-185a40c5d707,42,PV-ANLAGEN,TECHNOLOGY
92,ade05329-22e9-459a-86ef-fad8af25fda2,92,PV-ANLAGE,TECHNOLOGY
180,b17b43ce-7c2f-4318-a096-7f261d88149e,180,PHOTOVOLTAIKANLAGEN,TECHNOLOGY
211,3030e7ec-5e56-4df0-a537-ccf0b624719f,211,FOTOVOLTAIKANLAGE,TECHNOLOGY


In [8]:
from pathlib import Path

canonical_path = Path("../../kg/controlled_kg/canonical_entities.csv")

first_canonical_entity = pd.DataFrame([
    {
        "canonical_id": "TEC_0001",
        "canonical_label": "Photovoltaikanlage",
        "entity_type": "TECHNOLOGY",
        "aliases": "PV-ANLAGE; PV-ANLAGEN; PHOTOVOLTAIKANLAGEN; FOTOVOLTAIKANLAGE",
        "raw_entity_ids": (
            "3b926070-fa35-4f04-ba30-185a40c5d707; "
            "ade05329-22e9-459a-86ef-fad8af25fda2; "
            "b17b43ce-7c2f-4318-a096-7f261d88149e; "
            "3030e7ec-5e56-4df0-a537-ccf0b624719f"
        ),
        "description": "Photovoltaic installation or system used to generate electricity from solar energy.",
        "validation_status": "CORRECTED",
        "notes": "Merged four raw GraphRAG nodes representing singular/plural, abbreviation, and spelling variants of the same generic technology."
    }
])

first_canonical_entity

,canonical_id,canonical_label,entity_type,aliases,raw_entity_ids,description,validation_status,notes
0,TEC_0001,Photovoltaikanlage,TECHNOLOGY,PV-ANLAGE; PV-ANLAGEN; PHOTOVOLTAIKANLAGEN; FO...,3b926070-fa35-4f04-ba30-185a40c5d707; ade05329...,Photovoltaic installation or system used to ge...,CORRECTED,Merged four raw GraphRAG nodes representing si...


In [17]:
working_path = canonical_path.parent / "canonical_entities_working.csv"

first_canonical_entity.to_csv(
    working_path,
    index=False
)

print("Saved to:", working_path)

Saved to: ..\..\kg\controlled_kg\canonical_entities_working.csv


In [18]:
canonical_entities_working = pd.read_csv(working_path)
canonical_entities_working

,canonical_id,canonical_label,entity_type,aliases,raw_entity_ids,description,validation_status,notes
0,TEC_0001,Photovoltaikanlage,TECHNOLOGY,PV-ANLAGE; PV-ANLAGEN; PHOTOVOLTAIKANLAGEN; FO...,3b926070-fa35-4f04-ba30-185a40c5d707; ade05329...,Photovoltaic installation or system used to ge...,CORRECTED,Merged four raw GraphRAG nodes representing si...


In [19]:
agri_pv_ids = [93, 245]

agri_pv = entities[
    entities["human_readable_id"].isin(agri_pv_ids)
]

for _, row in agri_pv.iterrows():
    print("=" * 80)
    print("ID:", row["human_readable_id"])
    print("TITLE:", row["title"])
    print("TYPE:", row["type"])
    print("DESCRIPTION:")
    print(row["description"])
    print("TEXT UNIT IDS:")
    print(row["text_unit_ids"])
    print()

ID: 93
TITLE: AGRI-PV-ANLAGE
TYPE: TECHNOLOGY
DESCRIPTION:
AGRI-PV-ANLAGE refers to a form of photovoltaic installation integrated into agriculture that enables dual use of agricultural land for both farming and electricity production, allowing continued predominant agricultural activity on the same area. This type of photovoltaic system is referenced for its potential contributions to acceptance, biodiversity, enhanced land-use efficiency, and providing new income opportunities for agriculture.
TEXT UNIT IDS:
['e498566a78d36b3ce13304157993b9b4269045840f1d63663ab07328cc5c6a9ebc15f47c9ae340cbebf706bc8faebd35ccc4f11ae0aa4cb9a223345224329757'
 '23663988970639bea940978ccbd9150fd292856f2f31a1d587392606a64713d5dfeade66f1beea67d1cc0b0e93012337685e0eca3267f98c1695a466657a8b24'
 '4d221520fe7090409cfa9cda598db30683356c340ab3584a063ffe92486f2fe7225fc056f8b997f085ff0c0f7475828879fa678b7b1709148cb03584b292dab0']

ID: 245
TITLE: AGRI-PV-ANLAGEN
TYPE: TECHNOLOGY
DESCRIPTION:
Agri-PV systems are insta

In [20]:
agri_pv[
    [
        "id",
        "human_readable_id",
        "title",
        "type"
    ]
]

,id,human_readable_id,title,type
93,077391bf-ba15-4ade-9a43-449741d7f5be,93,AGRI-PV-ANLAGE,TECHNOLOGY
245,10850660-c7bd-4fcf-bb7d-77add2e98cd0,245,AGRI-PV-ANLAGEN,TECHNOLOGY


In [21]:
second_canonical_entity = pd.DataFrame([
    {
        "canonical_id": "TEC_0002",
        "canonical_label": "Agri-PV-Anlage",
        "entity_type": "TECHNOLOGY",
        "aliases": "AGRI-PV-ANLAGE; AGRI-PV-ANLAGEN",
        "raw_entity_ids": (
            "077391bf-ba15-4ade-9a43-449741d7f5be; "
            "10850660-c7bd-4fcf-bb7d-77add2e98cd0"
        ),
        "description": (
            "Photovoltaic installation enabling combined agricultural land use "
            "and electricity generation."
        ),
        "validation_status": "CORRECTED",
        "notes": (
            "Merged singular and plural GraphRAG nodes referring to the same "
            "Agri-PV technology; kept separate from the generic Photovoltaikanlage entity."
        )
    }
])

second_canonical_entity

,canonical_id,canonical_label,entity_type,aliases,raw_entity_ids,description,validation_status,notes
0,TEC_0002,Agri-PV-Anlage,TECHNOLOGY,AGRI-PV-ANLAGE; AGRI-PV-ANLAGEN,077391bf-ba15-4ade-9a43-449741d7f5be; 10850660...,Photovoltaic installation enabling combined ag...,CORRECTED,Merged singular and plural GraphRAG nodes refe...


In [22]:
canonical_entities_working = pd.read_csv(working_path)

if "TEC_0002" not in canonical_entities_working["canonical_id"].values:
    canonical_entities_working = pd.concat(
        [canonical_entities_working, second_canonical_entity],
        ignore_index=True
    )

    canonical_entities_working.to_csv(
        working_path,
        index=False
    )

    print("TEC_0002 saved successfully.")
else:
    print("TEC_0002 already exists.")

canonical_entities_working

TEC_0002 saved successfully.


,canonical_id,canonical_label,entity_type,aliases,raw_entity_ids,description,validation_status,notes
0,TEC_0001,Photovoltaikanlage,TECHNOLOGY,PV-ANLAGE; PV-ANLAGEN; PHOTOVOLTAIKANLAGEN; FO...,3b926070-fa35-4f04-ba30-185a40c5d707; ade05329...,Photovoltaic installation or system used to ge...,CORRECTED,Merged four raw GraphRAG nodes representing si...
1,TEC_0002,Agri-PV-Anlage,TECHNOLOGY,AGRI-PV-ANLAGE; AGRI-PV-ANLAGEN,077391bf-ba15-4ade-9a43-449741d7f5be; 10850660...,Photovoltaic installation enabling combined ag...,CORRECTED,Merged singular and plural GraphRAG nodes refe...


In [23]:
ministry_candidates = entities[
    (entities["type"] == "ORGANIZATION") &
    (
        entities["title"].str.contains(
            r"BMK|BUNDESMINISTERIUM|MINISTERIUM",
            case=False,
            na=False,
            regex=True
        )
    )
].copy()

ministry_candidates[
    [
        "human_readable_id",
        "title",
        "type",
        "frequency",
        "degree"
    ]
].sort_values("title")

,human_readable_id,title,type,frequency,degree
33,33,BMK,ORGANIZATION,3,3
537,537,"BUNDESMINISTERIUM FÜR KLIMASCHUTZ, UMWELT, ENE...",ORGANIZATION,1,1
1,1,"BUNDESMINISTERIUM FÜR KLIMASCHUTZ, UMWELT, ENE...",ORGANIZATION,2,6


In [24]:
ministry_ids = [33, 537, 1]

ministry_group = entities[
    entities["human_readable_id"].isin(ministry_ids)
]

for _, row in ministry_group.sort_values("human_readable_id").iterrows():
    print("=" * 100)
    print("HUMAN READABLE ID:", row["human_readable_id"])
    print("GRAPH ID:", row["id"])
    print("TITLE:")
    print(row["title"])
    print("TYPE:", row["type"])
    print("DESCRIPTION:")
    print(row["description"])
    print("TEXT UNIT IDS:")
    print(row["text_unit_ids"])
    print()

HUMAN READABLE ID: 1
GRAPH ID: 86fa5f94-0db2-4a5f-8d24-d32bb067610b
TITLE:
BUNDESMINISTERIUM FÜR KLIMASCHUTZ, UMWELT, ENERGIE, MOBILITÄT, INNOVATION UND TECHNOLOGIE (BMK)
TYPE: ORGANIZATION
DESCRIPTION:
The BUNDESMINISTERIUM FÜR KLIMASCHUTZ, UMWELT, ENERGIE, MOBILITÄT, INNOVATION UND TECHNOLOGIE (BMK), also known in English as the Austrian Federal Ministry for Climate Action, Environment, Energy, Mobility, Innovation and Technology, is responsible for climate, environmental, and energy policy, and is referenced as the publisher or commissioner of several policy documents and statistical market reports, including serving as the publisher and initiator of the Austrian Photovoltaic Strategy.
TEXT UNIT IDS:
['d7219f76b31818b7b13bf08dda81003d752e53f7660831d5e201d105d2bc1fc19e8cc49ca6cd21e08570939d7fb3e87b45f0f5feb8575df195e7be8cbd3b6475'
 '6927535991c09031efd0df39ec21591dd3bb460da14c01bc36d343f0078b726ed44d06a0c6bb518236ea61ed8c8b15365630cf185b8012e963898f2e117f7d8a']

HUMAN READABLE ID: 33

In [25]:
third_canonical_entity = pd.DataFrame([
    {
        "canonical_id": "ORG_0001",
        "canonical_label": (
            "Bundesministerium für Klimaschutz, Umwelt, Energie, "
            "Mobilität, Innovation und Technologie (BMK)"
        ),
        "entity_type": "ORGANIZATION",
        "aliases": (
            "BMK; "
            "BUNDESMINISTERIUM FÜR KLIMASCHUTZ, UMWELT, ENERGIE, "
            "MOBILITÄT, INNOVATION UND TECHNOLOGIE"
        ),
        "raw_entity_ids": (
            "86fa5f94-0db2-4a5f-8d24-d32bb067610b; "
            "70ab370d-ec2e-432b-9a52-b46cbdf6074f; "
            "fe95369a-1060-4811-8237-6ecb9da9300d"
        ),
        "description": (
            "Austrian federal ministry responsible for climate, "
            "environmental and energy policy in the source document."
        ),
        "validation_status": "CORRECTED",
        "notes": (
            "Merged acronym BMK and two full-name GraphRAG nodes "
            "that refer to the same organization."
        )
    }
])

third_canonical_entity

,canonical_id,canonical_label,entity_type,aliases,raw_entity_ids,description,validation_status,notes
0,ORG_0001,"Bundesministerium für Klimaschutz, Umwelt, Ene...",ORGANIZATION,"BMK; BUNDESMINISTERIUM FÜR KLIMASCHUTZ, UMWELT...",86fa5f94-0db2-4a5f-8d24-d32bb067610b; 70ab370d...,Austrian federal ministry responsible for clim...,CORRECTED,Merged acronym BMK and two full-name GraphRAG ...


In [26]:
canonical_entities_working = pd.read_csv(working_path)

if "ORG_0001" not in canonical_entities_working["canonical_id"].values:
    canonical_entities_working = pd.concat(
        [canonical_entities_working, third_canonical_entity],
        ignore_index=True
    )

    canonical_entities_working.to_csv(
        working_path,
        index=False
    )

    print("ORG_0001 saved successfully.")
else:
    print("ORG_0001 already exists.")

canonical_entities_working

ORG_0001 saved successfully.


,canonical_id,canonical_label,entity_type,aliases,raw_entity_ids,description,validation_status,notes
0,TEC_0001,Photovoltaikanlage,TECHNOLOGY,PV-ANLAGE; PV-ANLAGEN; PHOTOVOLTAIKANLAGEN; FO...,3b926070-fa35-4f04-ba30-185a40c5d707; ade05329...,Photovoltaic installation or system used to ge...,CORRECTED,Merged four raw GraphRAG nodes representing si...
1,TEC_0002,Agri-PV-Anlage,TECHNOLOGY,AGRI-PV-ANLAGE; AGRI-PV-ANLAGEN,077391bf-ba15-4ade-9a43-449741d7f5be; 10850660...,Photovoltaic installation enabling combined ag...,CORRECTED,Merged singular and plural GraphRAG nodes refe...
2,ORG_0001,"Bundesministerium für Klimaschutz, Umwelt, Ene...",ORGANIZATION,"BMK; BUNDESMINISTERIUM FÜR KLIMASCHUTZ, UMWELT...",86fa5f94-0db2-4a5f-8d24-d32bb067610b; 70ab370d...,Austrian federal ministry responsible for clim...,CORRECTED,Merged acronym BMK and two full-name GraphRAG ...


In [27]:
policy_candidates = entities[
    (entities["type"] == "POLICY") &
    (
        entities["title"].str.contains(
            r"PHOTOVOLTAIK.*STRATEG|PV.*STRATEG|STRATEG.*PHOTOVOLTAIK",
            case=False,
            na=False,
            regex=True
        )
    )
].copy()

policy_candidates[
    [
        "human_readable_id",
        "title",
        "type",
        "frequency",
        "degree"
    ]
].sort_values("title")

,human_readable_id,title,type,frequency,degree
31,31,PHOTOVOLTAIK-STRATEGIE,POLICY,1,21
6,6,ÖSTERREICHISCHE PHOTOVOLTAIK-STRATEGIE,POLICY,10,45


In [28]:
policy_ids = [31, 6]

policy_group = entities[
    entities["human_readable_id"].isin(policy_ids)
]

for _, row in policy_group.sort_values("human_readable_id").iterrows():
    print("=" * 100)
    print("HUMAN READABLE ID:", row["human_readable_id"])
    print("GRAPH ID:", row["id"])
    print("TITLE:", row["title"])
    print("TYPE:", row["type"])
    print("DESCRIPTION:")
    print(row["description"])
    print("TEXT UNIT IDS:")
    print(row["text_unit_ids"])
    print()

HUMAN READABLE ID: 6
GRAPH ID: fac405b8-06d4-43da-b1fc-6e707fd0cade
TITLE: ÖSTERREICHISCHE PHOTOVOLTAIK-STRATEGIE
TYPE: POLICY
DESCRIPTION:
The "ÖSTERREICHISCHE PHOTOVOLTAIK-STRATEGIE" (Austrian Photovoltaic Strategy) is a strategy policy for Austria that provides a framework for the expansion, market development, innovation, and social acceptance of photovoltaic (PV) technologies. It outlines framework conditions, targets, objectives, vision, measures, and strategic action fields for coordinated national PV expansion, and includes guidelines and directives for the optimal planning and support of technical, economic, regulatory, and social aspects of PV expansion. The strategy is described as a detailed planning document and a "living" document, intended to be adapted to changing developments, and is relevant to supporting national research, workforce development, and sector-wide strategic PV expansion up to 2040. Its implementation status is described as in force or ongoing in some so

In [29]:
fourth_canonical_entity = pd.DataFrame([
    {
        "canonical_id": "POL_0001",
        "canonical_label": "Österreichische Photovoltaik-Strategie",
        "entity_type": "POLICY",
        "aliases": (
            "PHOTOVOLTAIK-STRATEGIE; "
            "ÖSTERREICHISCHE PHOTOVOLTAIK-STRATEGIE"
        ),
        "raw_entity_ids": (
            "fac405b8-06d4-43da-b1fc-6e707fd0cade; "
            "7104f7f4-94d3-4ffe-aa62-9b33ba3db307"
        ),
        "description": (
            "Austrian national strategy providing a framework for "
            "photovoltaic expansion and market development."
        ),
        "validation_status": "CORRECTED",
        "notes": (
            "Merged short and full GraphRAG names referring to the same "
            "Austrian Photovoltaic Strategy."
        )
    }
])

fourth_canonical_entity

,canonical_id,canonical_label,entity_type,aliases,raw_entity_ids,description,validation_status,notes
0,POL_0001,Österreichische Photovoltaik-Strategie,POLICY,PHOTOVOLTAIK-STRATEGIE; ÖSTERREICHISCHE PHOTOV...,fac405b8-06d4-43da-b1fc-6e707fd0cade; 7104f7f4...,Austrian national strategy providing a framewo...,CORRECTED,Merged short and full GraphRAG names referring...


In [30]:
canonical_entities_working = pd.read_csv(working_path)

if "POL_0001" not in canonical_entities_working["canonical_id"].values:
    canonical_entities_working = pd.concat(
        [canonical_entities_working, fourth_canonical_entity],
        ignore_index=True
    )

    canonical_entities_working.to_csv(
        working_path,
        index=False
    )

    print("POL_0001 saved successfully.")
else:
    print("POL_0001 already exists.")

canonical_entities_working

POL_0001 saved successfully.


,canonical_id,canonical_label,entity_type,aliases,raw_entity_ids,description,validation_status,notes
0,TEC_0001,Photovoltaikanlage,TECHNOLOGY,PV-ANLAGE; PV-ANLAGEN; PHOTOVOLTAIKANLAGEN; FO...,3b926070-fa35-4f04-ba30-185a40c5d707; ade05329...,Photovoltaic installation or system used to ge...,CORRECTED,Merged four raw GraphRAG nodes representing si...
1,TEC_0002,Agri-PV-Anlage,TECHNOLOGY,AGRI-PV-ANLAGE; AGRI-PV-ANLAGEN,077391bf-ba15-4ade-9a43-449741d7f5be; 10850660...,Photovoltaic installation enabling combined ag...,CORRECTED,Merged singular and plural GraphRAG nodes refe...
2,ORG_0001,"Bundesministerium für Klimaschutz, Umwelt, Ene...",ORGANIZATION,"BMK; BUNDESMINISTERIUM FÜR KLIMASCHUTZ, UMWELT...",86fa5f94-0db2-4a5f-8d24-d32bb067610b; 70ab370d...,Austrian federal ministry responsible for clim...,CORRECTED,Merged acronym BMK and two full-name GraphRAG ...
3,POL_0001,Österreichische Photovoltaik-Strategie,POLICY,PHOTOVOLTAIK-STRATEGIE; ÖSTERREICHISCHE PHOTOV...,fac405b8-06d4-43da-b1fc-6e707fd0cade; 7104f7f4...,Austrian national strategy providing a framewo...,CORRECTED,Merged short and full GraphRAG names referring...


In [31]:
rules_path = canonical_path.parent / "normalization_rules.md"

rules_text = """# Controlled KG Normalization Rules

## Purpose

These rules define how raw GraphRAG entities are mapped to canonical entities
in the controlled Austrian solar-market Knowledge Graph.

## Rule 1 — Singular and plural variants

Merge singular and plural forms when they refer to the same underlying concept.

Example:
- PV-ANLAGE
- PV-ANLAGEN
- PHOTOVOLTAIKANLAGEN
- FOTOVOLTAIKANLAGE

Canonical entity:
- TEC_0001 — Photovoltaikanlage

## Rule 2 — Acronym and full organization name

Merge an acronym with its full name when the descriptions and source context
confirm that they refer to the same organization.

Example:
- BMK
- Bundesministerium für Klimaschutz, Umwelt, Energie, Mobilität,
  Innovation und Technologie

Canonical entity:
- ORG_0001

## Rule 3 — Short and full policy names

Merge short and full policy names when they clearly refer to the same policy.

Example:
- PHOTOVOLTAIK-STRATEGIE
- ÖSTERREICHISCHE PHOTOVOLTAIK-STRATEGIE

Canonical entity:
- POL_0001

## Rule 4 — Do not merge related but distinct concepts

Do not merge entities merely because they are closely related.

Example:
- Photovoltaikanlage
- Agri-PV-Anlage

These remain separate because Agri-PV is a more specific technology concept.

Canonical entities:
- TEC_0001 — Photovoltaikanlage
- TEC_0002 — Agri-PV-Anlage

## Rule 5 — Evidence before merging

Do not merge entities based only on similar names.

Before merging, compare:
- entity type
- full title
- description
- source text-unit IDs
- source context where necessary

Decision values:
- MERGE
- KEEP SEPARATE
- UNCERTAIN

## Rule 6 — Preserve provenance

Every canonical entity must preserve the raw GraphRAG entity IDs from which
it was created.

Raw IDs must not be discarded because they are needed later to redirect
relationships and trace normalized entities back to GraphRAG output.

## Rule 7 — Conservative canonical descriptions

Canonical descriptions should describe the identity of the entity.

Do not copy every contextual claim from the GraphRAG-generated description
into the canonical description. Claims about targets, responsibilities,
requirements, status, or effects should later be represented as relationships
or properties when supported by evidence.
"""

rules_path.write_text(rules_text, encoding="utf-8")

print("Saved:", rules_path)

Saved: ..\..\kg\controlled_kg\normalization_rules.md


In [32]:
relationships_path = "../output/relationships.parquet"

relationships = pd.read_parquet(relationships_path)

print("Number of relationships:", len(relationships))
print("Columns:", relationships.columns.tolist())

Number of relationships: 643
Columns: ['id', 'human_readable_id', 'source', 'target', 'description', 'weight', 'combined_degree', 'text_unit_ids']


In [33]:
org_policy_relationships = relationships[
    (
        relationships["source"].str.contains(
            r"BMK|BUNDESMINISTERIUM",
            case=False,
            na=False,
            regex=True
        )
        &
        relationships["target"].str.contains(
            r"PHOTOVOLTAIK.*STRATEG",
            case=False,
            na=False,
            regex=True
        )
    )
    |
    (
        relationships["target"].str.contains(
            r"BMK|BUNDESMINISTERIUM",
            case=False,
            na=False,
            regex=True
        )
        &
        relationships["source"].str.contains(
            r"PHOTOVOLTAIK.*STRATEG",
            case=False,
            na=False,
            regex=True
        )
    )
].copy()

org_policy_relationships[
    [
        "human_readable_id",
        "source",
        "target",
        "description",
        "weight",
        "text_unit_ids"
    ]
]

,human_readable_id,source,target,description,weight,text_unit_ids
0,0,"BUNDESMINISTERIUM FÜR KLIMASCHUTZ, UMWELT, ENE...",ÖSTERREICHISCHE PHOTOVOLTAIK-STRATEGIE,[RELATION_TYPE=RESPONSIBLE_FOR] [MODALITY=EXPL...,10.0,[d7219f76b31818b7b13bf08dda81003d752e53f766083...
642,642,"BUNDESMINISTERIUM FÜR KLIMASCHUTZ, UMWELT, ENE...",ÖSTERREICHISCHE PHOTOVOLTAIK-STRATEGIE,[RELATION_TYPE=RESPONSIBLE_FOR] [MODALITY=REAS...,6.0,[7ae981575e185e2452d144d541d822b7725e27bc7dda5...


In [34]:
relationship_ids = [0, 642]

relationship_group = relationships[
    relationships["human_readable_id"].isin(relationship_ids)
]

for _, row in relationship_group.sort_values("human_readable_id").iterrows():
    print("=" * 100)
    print("HUMAN READABLE ID:", row["human_readable_id"])
    print("GRAPH RELATIONSHIP ID:", row["id"])
    print("SOURCE:")
    print(row["source"])
    print("TARGET:")
    print(row["target"])
    print("DESCRIPTION:")
    print(row["description"])
    print("WEIGHT:", row["weight"])
    print("TEXT UNIT IDS:")
    print(row["text_unit_ids"])
    print()

HUMAN READABLE ID: 0
GRAPH RELATIONSHIP ID: 766acd95-2d5a-4286-bf19-3455bb2616ec
SOURCE:
BUNDESMINISTERIUM FÜR KLIMASCHUTZ, UMWELT, ENERGIE, MOBILITÄT, INNOVATION UND TECHNOLOGIE (BMK)
TARGET:
ÖSTERREICHISCHE PHOTOVOLTAIK-STRATEGIE
DESCRIPTION:
[RELATION_TYPE=RESPONSIBLE_FOR] [MODALITY=EXPLICIT_FACT] The BMK is listed as the publisher and issuer of the Austrian Photovoltaic Strategy.
WEIGHT: 10.0
TEXT UNIT IDS:
['d7219f76b31818b7b13bf08dda81003d752e53f7660831d5e201d105d2bc1fc19e8cc49ca6cd21e08570939d7fb3e87b45f0f5feb8575df195e7be8cbd3b6475']

HUMAN READABLE ID: 642
GRAPH RELATIONSHIP ID: 0d51bb6b-e917-4703-b77d-aef2fc025f29
SOURCE:
BUNDESMINISTERIUM FÜR KLIMASCHUTZ, UMWELT, ENERGIE, MOBILITÄT, INNOVATION UND TECHNOLOGIE
TARGET:
ÖSTERREICHISCHE PHOTOVOLTAIK-STRATEGIE
DESCRIPTION:
[RELATION_TYPE=RESPONSIBLE_FOR] [MODALITY=REASONABLE_INFERENCE] The Federal Ministry for Climate Action, Environment, Energy, Mobility, Innovation and Technology is the government body responsible for the Austr

In [35]:
first_normalized_relationship = pd.DataFrame([
    {
        "relationship_id": "REL_0001",
        "source_id": "ORG_0001",
        "predicate": "RESPONSIBLE_FOR",
        "target_id": "POL_0001",
        "modality": "EXPLICIT_FACT",
        "status": "",
        "raw_relationship_ids": (
            "766acd95-2d5a-4286-bf19-3455bb2616ec; "
            "0d51bb6b-e917-4703-b77d-aef2fc025f29"
        ),
        "text_unit_ids": (
            "d7219f76b31818b7b13bf08dda81003d752e53f7660831d5e201d105d2bc1fc19e8cc49ca6cd21e08570939d7fb3e87b45f0f5feb8575df195e7be8cbd3b6475; "
            "7ae981575e185e2452d144d541d822b7725e27bc7dda501c9d8495719003ccb89584fe5e1ac83baa67ae4060497ae5e2536fae015a551dc8cdefa9b5aa17289b"
        ),
        "validation_status": "CORRECTED",
        "normalization_note": (
            "Collapsed two duplicate GraphRAG relationships after entity canonicalization. "
            "Used EXPLICIT_FACT because one source explicitly identifies BMK as publisher "
            "and issuer; retained the second inferential extraction as additional provenance."
        )
    }
])

first_normalized_relationship

,relationship_id,source_id,predicate,target_id,modality,status,raw_relationship_ids,text_unit_ids,validation_status,normalization_note
0,REL_0001,ORG_0001,RESPONSIBLE_FOR,POL_0001,EXPLICIT_FACT,,766acd95-2d5a-4286-bf19-3455bb2616ec; 0d51bb6b...,d7219f76b31818b7b13bf08dda81003d752e53f7660831...,CORRECTED,Collapsed two duplicate GraphRAG relationships...


In [36]:
normalized_relationships_path = (
    canonical_path.parent / "normalized_relationships_working.csv"
)

first_normalized_relationship.to_csv(
    normalized_relationships_path,
    index=False
)

print("Saved:", normalized_relationships_path)

Saved: ..\..\kg\controlled_kg\normalized_relationships_working.csv


In [37]:
normalized_relationships_working = pd.read_csv(
    normalized_relationships_path
)

normalized_relationships_working

,relationship_id,source_id,predicate,target_id,modality,status,raw_relationship_ids,text_unit_ids,validation_status,normalization_note
0,REL_0001,ORG_0001,RESPONSIBLE_FOR,POL_0001,EXPLICIT_FACT,NaN,766acd95-2d5a-4286-bf19-3455bb2616ec; 0d51bb6b...,d7219f76b31818b7b13bf08dda81003d752e53f7660831...,CORRECTED,Collapsed two duplicate GraphRAG relationships...


In [38]:
agri_relationships = relationships[
    relationships["source"].str.contains(
        r"AGRI-PV", case=False, na=False, regex=True
    )
    |
    relationships["target"].str.contains(
        r"AGRI-PV", case=False, na=False, regex=True
    )
].copy()

agri_relationships[
    [
        "human_readable_id",
        "source",
        "target",
        "description",
        "weight",
        "text_unit_ids"
    ]
]

,human_readable_id,source,target,description,weight,text_unit_ids
113,113,BIODIVERSITÄTS-SOLARPARK,KOSTENGÜNSTIGER STROM AUS FREIFLÄCHEN-BIODIVER...,[RELATION_TYPE=MEASURED_BY] [MODALITY=EXPLICIT...,10.0,[e498566a78d36b3ce13304157993b9b4269045840f1d6...
114,114,AGRI-PV-ANLAGE,KOSTENGÜNSTIGER STROM AUS FREIFLÄCHEN-BIODIVER...,[RELATION_TYPE=MEASURED_BY] [MODALITY=EXPLICIT...,10.0,[e498566a78d36b3ce13304157993b9b4269045840f1d6...
136,136,AGRI-PV-ANLAGE,LANDWIRT:INNEN,[RELATION_TYPE=SUPPORTS] [MODALITY=EXPLICIT_FA...,9.0,[e498566a78d36b3ce13304157993b9b4269045840f1d6...
165,165,ERNEUERBAREN-AUSBAU-GESETZ (EAG),INVESTITIONSZUSCHUSS FÜR AGRI-PV ANLAGEN,[RELATION_TYPE=PROPOSES] [MODALITY=EXPLICIT_FA...,10.0,[f9e54ba797ff2b80a5f0e5f24e3901236ac4130e4e84d...
166,166,INVESTITIONSZUSCHUSS FÜR AGRI-PV ANLAGEN,PHOTOVOLTAIK,[RELATION_TYPE=SUPPORTS] [MODALITY=EXPLICIT_FA...,9.0,[f9e54ba797ff2b80a5f0e5f24e3901236ac4130e4e84d...
257,257,AGRI-PV-ANLAGE,FOTOVOLTAIKANLAGE,[RELATION_TYPE=PART_OF] [MODALITY=EXPLICIT_FAC...,10.0,[23663988970639bea940978ccbd9150fd292856f2f31a...
258,258,AGRI-PV-ANLAGE,LANDWIRTSCHAFT,[RELATION_TYPE=ASSOCIATED_WITH] [MODALITY=EXPL...,8.0,[23663988970639bea940978ccbd9150fd292856f2f31a...
280,280,DOPPELNUTZUNG MIT AGRI-PV,AGRI-PV-ANLAGE,[RELATION_TYPE=ALIAS_OF] [MODALITY=EXPLICIT_FA...,10.0,[23663988970639bea940978ccbd9150fd292856f2f31a...
281,281,LANDWIRTSCHAFT,AGRI-PV-ANLAGE,[RELATION_TYPE=BENEFITS_FROM] [MODALITY=EXPLIC...,9.0,[23663988970639bea940978ccbd9150fd292856f2f31a...
292,292,AGRI-PV-ANLAGEN,LANDWIRTSCHAFT,[RELATION_TYPE=SUPPORTS] [MODALITY=EXPLICIT_FA...,9.0,[41bd25afdc1cce74b36dd6db5e44e91354172fda16475...


In [39]:
relation_257 = relationships[
    relationships["human_readable_id"] == 257
]

for _, row in relation_257.iterrows():
    print("GRAPH RELATIONSHIP ID:", row["id"])
    print("SOURCE:", row["source"])
    print("TARGET:", row["target"])
    print("DESCRIPTION:")
    print(row["description"])
    print("WEIGHT:", row["weight"])
    print("TEXT UNIT IDS:")
    print(row["text_unit_ids"])

GRAPH RELATIONSHIP ID: 1b0a059f-694c-4e24-a3c7-766ebedc83a4
SOURCE: AGRI-PV-ANLAGE
TARGET: FOTOVOLTAIKANLAGE
DESCRIPTION:
[RELATION_TYPE=PART_OF] [MODALITY=EXPLICIT_FACT] Agri-PV is a specific application of photovoltaic systems within agriculture.
WEIGHT: 10.0
TEXT UNIT IDS:
['23663988970639bea940978ccbd9150fd292856f2f31a1d587392606a64713d5dfeade66f1beea67d1cc0b0e93012337685e0eca3267f98c1695a466657a8b24']


In [40]:
text_units_path = "../output/text_units.parquet"

text_units = pd.read_parquet(text_units_path)

print("Number of text units:", len(text_units))
print("Columns:", text_units.columns.tolist())

Number of text units: 17
Columns: ['id', 'human_readable_id', 'text', 'n_tokens', 'document_id', 'entity_ids', 'relationship_ids', 'covariate_ids']


In [41]:
supporting_text_unit_id = relation_257.iloc[0]["text_unit_ids"][0]

evidence_257 = text_units[
    text_units["id"] == supporting_text_unit_id
]

print("TEXT UNIT ID:", supporting_text_unit_id)
print("DOCUMENT ID:", evidence_257.iloc[0]["document_id"])
print("\nSOURCE TEXT:\n")
print(evidence_257.iloc[0]["text"])

TEXT UNIT ID: 23663988970639bea940978ccbd9150fd292856f2f31a1d587392606a64713d5dfeade66f1beea67d1cc0b0e93012337685e0eca3267f98c1695a466657a8b24
DOCUMENT ID: 39ed356c790f07e222ffe8e84bfb78009537e51e126410c75214a9fcc06d3155e184eea660a41ea5501fc44e62b343cc79e3983f8e945e86282ec5e8c6bdb5cf

SOURCE TEXT:

 bevorzugte PV-An-

wendungen, sowie eine Aufteilung der Fördermittel in Größenklassen sind darin bereits ent-
halten.  Auch  die  Novellierung  des  Wohnungseigentumsgesetzes  im  Jänner  20227 hat  Er-
leichterungen für den Bau von Photovoltaikanlagen bei Reihenhäusern oder Einzelgebäu-

den gebracht. Das Anfang 2024 in Begutachtung befindliche Elektrizitätswirtschaftsgesetz

(ElWG) schafft neue und zeitgemäße Spielregeln für den Strommarkt. Durch mehr Transpa-

renz im Netz, neue Marktrollen und Maßnahmen für mehr Flexibilität wird ein wichtiger

Beitrag zur schnelleren Integration von erneuerbaren Energieanlagen geschaffen. Weiters

wird auch das in Vorbereitung stehende Erneuerbaren-Aus

In [42]:
relation_257_decision = pd.DataFrame([
    {
        "raw_relationship_id": "1b0a059f-694c-4e24-a3c7-766ebedc83a4",
        "raw_source": "AGRI-PV-ANLAGE",
        "raw_predicate": "PART_OF",
        "raw_target": "FOTOVOLTAIKANLAGE",
        "canonical_source_id": "TEC_0002",
        "canonical_target_id": "TEC_0001",
        "decision": "REJECTED",
        "reason": (
            "The source states that agricultural installations should be designed "
            "as Agri-PV installations, but does not state that an Agri-PV installation "
            "is PART_OF a photovoltaic installation. The extracted predicate therefore "
            "overstates the source evidence."
        ),
        "text_unit_id": (
            "23663988970639bea940978ccbd9150fd292856f2f31a1d587392606a64713d5"
            "dfeade66f1beea67d1cc0b0e93012337685e0eca3267f98c1695a466657a8b24"
        )
    }
])

relation_257_decision

,raw_relationship_id,raw_source,raw_predicate,raw_target,canonical_source_id,canonical_target_id,decision,reason,text_unit_id
0,1b0a059f-694c-4e24-a3c7-766ebedc83a4,AGRI-PV-ANLAGE,PART_OF,FOTOVOLTAIKANLAGE,TEC_0002,TEC_0001,REJECTED,The source states that agricultural installati...,23663988970639bea940978ccbd9150fd292856f2f31a1...
